<a href="https://colab.research.google.com/github/nguyenkyvi/Machine_Learning/blob/main/Lab04/Lab04_KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# 2.3. -*-
"""Bước 1: tự tính khoảng cách và tự tìm láng giềng, chưa dùng thư viện.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import numpy as np
import pandas as pd

# Ba bạn: A và C học gần như y hệt nhau, B học khác hẳn
ba_ban = {
    "A": np.array([5.0, 3.0, 4000.0]),
    "B": np.array([25.0, 9.0, 4100.0]),
    "C": np.array([6.0, 3.5, 7000.0]),
}


def kc(u, v):
    """Khoang cach Euclid giua hai diem."""
    return np.sqrt(((u - v) ** 2).sum())


print("Khoang cach KHI CHUA chuan hoa")
print(f"  d(A, B) = {kc(ba_ban['A'], ba_ban['B']):8.1f}")
print(f"  d(A, C) = {kc(ba_ban['A'], ba_ban['C']):8.1f}")
print()

# Chuẩn hoá bằng trung bình và độ lệch chuẩn của cả bộ dữ liệu
df = pd.read_csv("data/sinh_vien_mo_rong.csv")
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]].to_numpy(dtype=float)
tb = X.mean(axis=0)
dl = X.std(axis=0)
print("Do lech chuan cua ba cot:")
print(f"  gio_on {dl[0]:.2f}    diem_giua_ky {dl[1]:.2f}")
print(f"  chi_tieu {dl[2]:.0f}")
print()

chuan = {ten: (v - tb) / dl for ten, v in ba_ban.items()}
print("Khoang cach SAU KHI chuan hoa")
print(f"  d(A, B) = {kc(chuan['A'], chuan['B']):.2f}")
print(f"  d(A, C) = {kc(chuan['A'], chuan['C']):.2f}")
print()

# Tìm 5 láng giềng gần nhất của một bạn mới, chỉ dùng hai cột học tập
X2 = df[["gio_on", "diem_giua_ky"]].to_numpy(dtype=float)
y = df["qua_mon"].to_numpy()
tb2, dl2 = X2.mean(axis=0), X2.std(axis=0)
ban_moi = np.array([12.0, 6.5])
X2c = (X2 - tb2) / dl2
moi_c = (ban_moi - tb2) / dl2
khoang = np.sqrt(((X2c - moi_c) ** 2).sum(axis=1))
gan = np.argsort(khoang)[:5]

print("Ban moi: on 12 gio, diem giua ky 6.5")
print("Nam lang gieng gan nhat:")
for i in gan:
    ket = "qua" if y[i] == 1 else "rot"
    print(f"  gio {X2[i, 0]:5.1f}  diem {X2[i, 1]:4.1f}  "
          f"khoang cach {khoang[i]:.3f}  -> {ket}")
so_qua = int(y[gan].sum())

print(f"Bo phieu: {so_qua} qua, {5 - so_qua} rot")
print("Du doan:", "qua mon" if so_qua >= 3 else "rot mon")

Khoang cach KHI CHUA chuan hoa
  d(A, B) =    102.2
  d(A, C) =   3000.0

Do lech chuan cua ba cot:
  gio_on 7.82    diem_giua_ky 2.09
  chi_tieu 1692

Khoang cach SAU KHI chuan hoa
  d(A, B) = 3.85
  d(A, C) = 1.79

Ban moi: on 12 gio, diem giua ky 6.5
Nam lang gieng gan nhat:
  gio  10.5  diem  6.4  khoang cach 0.198  -> qua
  gio  12.5  diem  6.1  khoang cach 0.202  -> qua
  gio  14.0  diem  6.4  khoang cach 0.260  -> qua
  gio   9.5  diem  6.1  khoang cach 0.373  -> rot
  gio  10.5  diem  5.8  khoang cach 0.386  -> qua
Bo phieu: 4 qua, 1 rot
Du doan: qua mon


In [11]:
# 3.2 -*-
"""Bước 2: đo hậu quả của việc quên chuẩn hoá trên tập kiểm tra."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_mo_rong.csv")
y = df["qua_mon"]

X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
X_tr, X_te, y_tr, y_te = train_test_split(
    X3, y, test_size=0.25, random_state=17, stratify=y
)


def cham(mo_hinh, cot):
    mo_hinh.fit(X_tr[cot], y_tr)
    return accuracy_score(y_te, mo_hinh.predict(X_te[cot]))


def knn_chuan():
    """KNN co chuan hoa, gop lam mot khoi bang pipeline."""
    return make_pipeline(StandardScaler(), KNeighborsClassifier(7))


ba_cot = ["gio_on", "diem_giua_ky", "chi_tieu"]
hai_cot = ["gio_on", "diem_giua_ky"]

a = cham(KNeighborsClassifier(n_neighbors=7), ba_cot)
b = cham(knn_chuan(), ba_cot)
c = cham(knn_chuan(), hai_cot)

print(f"Doan bua ca lop qua          : {y_te.mean():.4f}")
print(f"Ba cot, khong chuan hoa       : {a:.4f}")
print(f"Ba cot, co chuan hoa          : {b:.4f}")
print(f"Bo cot chi_tieu, co chuan hoa : {c:.4f}")

Doan bua ca lop qua          : 0.6000
Ba cot, khong chuan hoa       : 0.5333
Ba cot, co chuan hoa          : 0.8667
Bo cot chi_tieu, co chuan hoa : 0.9000


In [12]:
# 4.2 -*-
"""Bước 3: chọn số láng giềng k bằng kiểm định chéo."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Hai cột x1 và x2 có thang đo gần nhau, độ lệch chuẩn chỉ chênh khoảng
# 1.6 lần, nên ở bộ này không chuẩn hoá cũng được.
print("Accuracy tren tap hoc va tap kiem tra theo k")
print("  k    tap hoc    tap kiem tra")
for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"  {k:3d}    {a_hoc:.4f}      {a_kt:.4f}")
print()

# Chọn k chỉ dựa trên tập học, không nhìn tập kiểm tra
print("Kiem dinh cheo 5 phan tren tap hoc:")
diem = {}
for k in range(1, 52, 2):
    diem[k] = cross_val_score(KNeighborsClassifier(n_neighbors=k),
                              X_tr, y_tr, cv=5).mean()
k_tot = max(diem, key=diem.get)
for k in [1, 5, 9, 13, 17, 25, 51]:
    dau = " <- tot nhat" if k == k_tot else ""
    print(f"  k = {k:2d}  diem trung binh {diem[k]:.4f}{dau}")
print()

# Chọn xong mới mở tập kiểm tra, và chỉ mở một lần
mh = KNeighborsClassifier(n_neighbors=k_tot).fit(X_tr, y_tr)
print(f"k tot nhat = {k_tot}")
acc = accuracy_score(y_te, mh.predict(X_te))
print(f"Accuracy tren tap kiem tra = {acc:.4f}")

Accuracy tren tap hoc va tap kiem tra theo k
  k    tap hoc    tap kiem tra
    1    1.0000      0.8000
    5    0.8756      0.9067
   13    0.8844      0.9200
   31    0.8667      0.8933
  101    0.8311      0.8533

Kiem dinh cheo 5 phan tren tap hoc:
  k =  1  diem trung binh 0.7778
  k =  5  diem trung binh 0.8444
  k =  9  diem trung binh 0.8444
  k = 13  diem trung binh 0.8667 <- tot nhat
  k = 17  diem trung binh 0.8533
  k = 25  diem trung binh 0.8622
  k = 51  diem trung binh 0.8533

k tot nhat = 13
Accuracy tren tap kiem tra = 0.9200


In [13]:
# 5. -*-
"""Bước 4: đặt KNN cạnh hai mô hình đã học trên dữ liệu có biên cong."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cac_mo_hinh = [
    ("Hoi quy logistic", LogisticRegression()),
    ("Naive Bayes     ", GaussianNB()),
    ("KNN, k = 13     ", KNeighborsClassifier(n_neighbors=13)),
]

print("Accuracy tren tap kiem tra, bo hai vang trang")
for ten, mh in cac_mo_hinh:
    mh.fit(X_tr, y_tr)
    print(f"  {ten}: {accuracy_score(y_te, mh.predict(X_te)):.4f}")

Accuracy tren tap kiem tra, bo hai vang trang
  Hoi quy logistic: 0.8533
  Naive Bayes     : 0.8533
  KNN, k = 13     : 0.9200


In [14]:
# 6.
# -*- coding: utf-8 -*-
"""Bước 5: KNN cho bài toán hồi quy, trên dữ liệu giá nhà của bài 1."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("data/gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

# Cùng cách chia với bài 1 để so sánh công bằng
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

knn = KNeighborsRegressor(n_neighbors=5).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)

print("MAE tren tap kiem tra (ty dong)")
mae_knn = mean_absolute_error(y_te, knn.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))
print(f"  KNN, k = 5 : {mae_knn:.4f}")
print(f"  Duong thang : {mae_duong:.4f}")
print()

can_moi = pd.DataFrame({"dien_tich": [60.0, 100.0, 150.0, 250.0]})
print("Du doan cho bon can moi (ty dong)")
print("  dien tich      KNN    duong thang")
for dt, a, b in zip(can_moi["dien_tich"], knn.predict(can_moi),
                    duong.predict(can_moi)):
    print(f"   {dt:6.0f} m2   {a:6.2f}    {b:6.2f}")
print()

# Năm láng giềng của căn 250 m2 chính là những căn rộng nhất đã thấy
_, chi_so = knn.kneighbors(pd.DataFrame({"dien_tich": [250.0]}))
lg = X_tr.iloc[chi_so[0]]["dien_tich"].tolist()
print("Nam lang gieng cua can 250 m2:", ", ".join(f"{v:.1f}" for v in lg))

MAE tren tap kiem tra (ty dong)
  KNN, k = 5 : 0.4030
  Duong thang : 0.2578

Du doan cho bon can moi (ty dong)
  dien tich      KNN    duong thang
       60 m2     4.95      5.12
      100 m2     8.32      8.24
      150 m2     9.32     12.13
      250 m2     9.32     19.91

Nam lang gieng cua can 250 m2: 117.5, 115.9, 111.7, 111.2, 104.2


7. BÀi tập

In [15]:
# Bài tập 1
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# 1. Đọc dữ liệu hai vầng trăng
df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]].to_numpy()
y = df["nhan"].to_numpy()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# 2. Hàm dự đoán KNN tự viết
def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    # Tính khoảng cách Euclid từ x_moi đến toàn bộ X_hoc
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))

    # Lấy chỉ số của k khoảng cách nhỏ nhất
    chi_so_k_gan = np.argsort(khoang_cach)[:k]

    # Lấy nhãn tương ứng của k láng giềng
    nhan_k_gan = y_hoc[chi_so_k_gan]

    # Bầu chọn nhãn xuất hiện nhiều nhất
    nhan_pho_bien = Counter(nhan_k_gan).most_common(1)[0][0]
    return nhan_pho_bien

# 3. Dự đoán cho toàn bộ tập kiểm tra
y_pred_tu_viet = [knn_du_doan(X_tr, y_tr, x, k=13) for x in X_te]
acc_tu_viet = accuracy_score(y_te, y_pred_tu_viet)

# 4. So sánh với KNeighborsClassifier của scikit-learn
sk_knn = KNeighborsClassifier(n_neighbors=13).fit(X_tr, y_tr)
acc_sklearn = accuracy_score(y_te, sk_knn.predict(X_te))

print(f"Accuracy tu viet (k=13) : {acc_tu_viet:.4f}")
print(f"Accuracy sklearn (k=13) : {acc_sklearn:.4f}")

Accuracy tu viet (k=13) : 0.9200
Accuracy sklearn (k=13) : 0.9200


In [16]:
# Bài tập 2
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Tìm k tốt nhất với metric='manhattan'
diem = {}
for k in range(1, 52, 2):
    knn_manhattan = KNeighborsClassifier(n_neighbors=k, metric="manhattan")
    diem[k] = cross_val_score(knn_manhattan, X_tr, y_tr, cv=5).mean()

k_tot_manhattan = max(diem, key=diem.get)

# Đánh giá trên tập kiểm tra
mh = KNeighborsClassifier(n_neighbors=k_tot_manhattan, metric="manhattan").fit(X_tr, y_tr)
acc_manhattan = accuracy_score(y_te, mh.predict(X_te))

print(f"k tot nhat (Manhattan) = {k_tot_manhattan}")
print(f"Accuracy tap kiem tra (Manhattan) = {acc_manhattan:.4f}")

# Nhận xét: Đổi cách đo khoảng cách từ Euclid sang Manhattan không làm kết quả khác đi nhiều
# do hai đặc trưng x1, x2 có thang đo tương đồng và bài toán trong không gian 2D khá đơn giản.

k tot nhat (Manhattan) = 27
Accuracy tap kiem tra (Manhattan) = 0.9067


In [17]:
# Bài tập 3
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_mo_rong.csv")

# 1. Đổi cột chi_tieu từ nghìn đồng sang triệu đồng
df["chi_tieu"] = df["chi_tieu"] / 1000.0

X = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
y = df["qua_mon"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# 2. Chạy KNN k=7 KHÔNG chuẩn hoá
knn_no_scale = KNeighborsClassifier(n_neighbors=7).fit(X_tr, y_tr)
acc_no_scale = accuracy_score(y_te, knn_no_scale.predict(X_te))

# 3. Chạy KNN k=7 CÓ chuẩn hoá
knn_with_scale = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)).fit(X_tr, y_tr)
acc_with_scale = accuracy_score(y_te, knn_with_scale.predict(X_te))

print(f"Accuracy khong chuan hoa (sau khi doi don vi) : {acc_no_scale:.4f}")
print(f"Accuracy co chuan hoa (sau khi doi don vi)    : {acc_with_scale:.4f}")

"""
Giải thích:
- Khi chưa đổi đơn vị, 'chi_tieu' nằm ở tầm hàng nghìn (4000-7000), áp đảo các cột còn lại (chỉ tầm 1-10) nên độ chính xác bị thấp (~0.5333).
- Khi chia cho 1000, giá trị 'chi_tieu' thu về tầm 4-7 (cùng tầm với gio_on, diem_giua_ky), khoảng cách Euclid không còn bị chi phối bởi chi_tieu nữa nên accuracy tăng vọt.
- Nếu CÓ chuẩn hoá (StandardScaler), kết quả KHÔNG bị thay đổi khi đổi đơn vị vì chuẩn hoá đã tự động đưa mọi biến về cùng độ lệch chuẩn = 1.
"""

Accuracy khong chuan hoa (sau khi doi don vi) : 0.8667
Accuracy co chuan hoa (sau khi doi don vi)    : 0.8667


"\nGiải thích:\n- Khi chưa đổi đơn vị, 'chi_tieu' nằm ở tầm hàng nghìn (4000-7000), áp đảo các cột còn lại (chỉ tầm 1-10) nên độ chính xác bị thấp (~0.5333).\n- Khi chia cho 1000, giá trị 'chi_tieu' thu về tầm 4-7 (cùng tầm với gio_on, diem_giua_ky), khoảng cách Euclid không còn bị chi phối bởi chi_tieu nữa nên accuracy tăng vọt.\n- Nếu CÓ chuẩn hoá (StandardScaler), kết quả KHÔNG bị thay đổi khi đổi đơn vị vì chuẩn hoá đã tự động đưa mọi biến về cùng độ lệch chuẩn = 1.\n"

In [18]:
# Bài tập 4
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print(f"{'k':>3s} | {'Uniform (Deu)':>15s} | {'Distance (Trong so)':>20s}")
print("-" * 45)

for k in [1, 13, 101]:
    # Trọng số đều (mặc định)
    knn_unif = KNeighborsClassifier(n_neighbors=k, weights="uniform").fit(X_tr, y_tr)
    acc_unif = accuracy_score(y_te, knn_unif.predict(X_te))

    # Trọng số theo khoảng cách
    knn_dist = KNeighborsClassifier(n_neighbors=k, weights="distance").fit(X_tr, y_tr)
    acc_dist = accuracy_score(y_te, knn_dist.predict(X_te))

    print(f"{k:3d} | {acc_unif:15.4f} | {acc_dist:20.4f}")

"""
Trả lời câu hỏi:
Với k = 101 (rất lớn), weights='distance' giúp mô hình BỚT CẦU THẢ hơn rất nhiều.
Vì khi k lớn,weights='uniform' sẽ lấy phiếu của cả các điểm ở xa, làm biên quyết định bị phẳng hoá/mờ đi.
Ngược lại, weights='distance' đánh trọng số rất nhỏ cho các điểm ở xa, tập trung phiếu vào các điểm thực sự gần nhất, bảo toàn được biên cong.
"""

  k |   Uniform (Deu) |  Distance (Trong so)
---------------------------------------------
  1 |          0.8000 |               0.8000
 13 |          0.9200 |               0.9333
101 |          0.8533 |               0.9067


"\nTrả lời câu hỏi:\nVới k = 101 (rất lớn), weights='distance' giúp mô hình BỚT CẦU THẢ hơn rất nhiều.\nVì khi k lớn,weights='uniform' sẽ lấy phiếu của cả các điểm ở xa, làm biên quyết định bị phẳng hoá/mờ đi.\nNgược lại, weights='distance' đánh trọng số rất nhỏ cho các điểm ở xa, tập trung phiếu vào các điểm thực sự gần nhất, bảo toàn được biên cong.\n"

In [19]:
# Bài tâpj 5
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("data/gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Tìm k tốt nhất từ 1 tới 20 bằng kiểm định chéo 5 phần
diem_mae = {}
for k in range(1, 21):
    scores = cross_val_score(
        KNeighborsRegressor(n_neighbors=k),
        X_tr, y_tr, cv=5, scoring="neg_mean_absolute_error"
    )
    # cross_val_score trả về neg_MAE nên cần đổi dấu (-) thành (+)
    diem_mae[k] = -scores.mean()

k_tot = min(diem_mae, key=diem_mae.get)
print(f"k tot nhat cho Hoi quy = {k_tot} (MAE CV = {diem_mae[k_tot]:.4f})")

# Đánh giá trên tập kiểm tra
knn_best = KNeighborsRegressor(n_neighbors=k_tot).fit(X_tr, y_tr)
mae_knn = mean_absolute_error(y_te, knn_best.predict(X_te))

line_reg = LinearRegression().fit(X_tr, y_tr)
mae_line = mean_absolute_error(y_te, line_reg.predict(X_te))

print(f"MAE tap kiem tra (KNN, k={k_tot}) : {mae_knn:.4f}")
print(f"MAE tap kiem tra (Duong thang) : {mae_line:.4f}")

if mae_knn < mae_line:
    print("-> MAE của KNN vượt trội hơn (nhỏ hơn) so với đường thẳng!")
else:
    print("-> Đường thẳng có kết quả tốt hơn hoặc tương đương KNN.")

k tot nhat cho Hoi quy = 2 (MAE CV = 0.3498)
MAE tap kiem tra (KNN, k=2) : 0.4867
MAE tap kiem tra (Duong thang) : 0.2578
-> Đường thẳng có kết quả tốt hơn hoặc tương đương KNN.


In [20]:
# Bài tập 6
# -*- coding: utf-8 -*-
import time
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)

co_mau_list = [1000, 10000, 100000]
ket_qua = []

# Tập dữ liệu mới để test thời gian predict (1000 mẫu)
X_moi = rng.random((1000, 2))

for n in co_mau_list:
    # Tạo tập huấn luyện
    X_train = rng.random((n, 2))
    y_train = rng.integers(0, 2, size=n)

    knn = KNeighborsClassifier(n_neighbors=5)

    # Đo thời gian fit
    t0 = time.time()
    knn.fit(X_train, y_train)
    t_fit = time.time() - t0

    # Đo thời gian predict cho 1000 mẫu mới
    t0 = time.time()
    knn.predict(X_moi)
    t_predict = time.time() - t0

    ket_qua.append({
        "Co mau (N)": n,
        "Thoi gian .fit() (s)": round(t_fit, 6),
        "Thoi gian .predict() (s)": round(t_predict, 6)
    })

df_ketqua = pd.DataFrame(ket_qua)
print(df_ketqua.to_string(index=False))

"""
Giải thích:
- KNN được gọi là "mô hình học lười" (Lazy Learning) vì bước huấn luyện (.fit) nó hầu như KHÔNG làm gì ngoài việc lưu lại bộ dữ liệu vào bộ nhớ (thời gian gần như bằng 0).
- Ngược lại, ở bước dự đoán (.predict), nó mới bắt đầu tính khoảng cách từ mẫu mới tới TOÀN BỘ N mẫu trong cơ sở dữ liệu. Vì vậy khi cỡ mẫu N tăng lên, thời gian .predict() sẽ tăng lên rõ rệt.
"""

 Co mau (N)  Thoi gian .fit() (s)  Thoi gian .predict() (s)
       1000              0.001958                  0.005297
      10000              0.006859                  0.005532
     100000              0.075082                  0.007956


'\nGiải thích:\n- KNN được gọi là "mô hình học lười" (Lazy Learning) vì bước huấn luyện (.fit) nó hầu như KHÔNG làm gì ngoài việc lưu lại bộ dữ liệu vào bộ nhớ (thời gian gần như bằng 0).\n- Ngược lại, ở bước dự đoán (.predict), nó mới bắt đầu tính khoảng cách từ mẫu mới tới TOÀN BỘ N mẫu trong cơ sở dữ liệu. Vì vậy khi cỡ mẫu N tăng lên, thời gian .predict() sẽ tăng lên rõ rệt.\n'